# Problem 2
## Code to support answers in this cell is shown below:

#### a) Identify best CCP pruning sets determined in 10 fold CV expirment and the value of the RMSE. Compare the RMSE of the pruned tree to the RMSE of the full tree.
- The best CCP pruning alpha is: `86032.7676560414`
- The RMSE of the pruned tree is: `7478.562978527341`
- The RMSE of the full tree is: `8093.7177697670895`
- The pruned tree has less error than the full tree and is way less complex given that the pruned tree is 140 leaves and the full tree is 4800 leaves.

#### b) Construct  regression Tree on all the training and validation data. Predict the sales price of a vehicle with values from the table. 
- A quick google shows that for the transmission column which was not supplied for this table, the 2016 Charger only came in automatic so I filled that in
- Predicted Price: `$21,390.50`

#### c) Construct a Random Forest Regressor with 10 fold CV and report best parameters and RMSE value
- The best Parameters are: `{'max_features': 0.33, 'min_samples_leaf': 1, 'n_estimators': 300}`
- The RMSE is: `5500.139827711852`

#### d) Construct a Gradient Boosting Regressor with 10 fold CV and report best parameters and RMSE value
- The best parameters are: `{'learning_rate': 0.02, 'max_depth': 5, 'n_estimators': 800}`
- The RMSE is: `5625.325449982929`

#### e) Compare the performance of the regressors to select a final model. Construct this on all the training/validation data and determine RMSE on test set. How does it compare to the train set RMSE?
- The best model was: `Random Forest Regressor` for the training and validation data.
- The RMSE on the test set is: `6450.283194968927`
- The The RMSE is about 1000 larger than the training set which is significant variance. However, I looked at the individual fold performances and it does fall within those. The thing that is really hurting model performance here is the outliers. Especially given that the outlier in the test set is larger than the outlier in the training set by a decent margin.

In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score, KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (root_mean_squared_error, mean_absolute_error)

palanquinloc = "data/palanquin.csv"

In [2]:
df = pd.read_csv(palanquinloc)

df.head()

,Model,Vehicle,Year,Manufacturer,Price,Condition,Cylinders,Fuel,Odometer,Transmission,Drive,Type,Paint,State
0,wrangler unlimited,8680,2016,jeep,33995,good,6,gas,48431,automatic,4wd,wagon,custom,ca
1,cooper countryman,10426,2018,mini,29000,like new,4,hybrid,17622,automatic,4wd,SUV,grey,co
2,grand cherokee limited 4x4,7243,1994,jeep,3950,like new,8,gas,222000,automatic,4wd,SUV,white,ca
3,genesis,6225,2012,hyundai,10900,like new,8,gas,112000,automatic,rwd,sedan,silver,ca
4,pre runner,4501,2001,toyota,8300,excellent,6,gas,264000,automatic,rwd,truck,red,ca


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10824 entries, 0 to 10823
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Model         10824 non-null  str  
 1   Vehicle       10824 non-null  int64
 2   Year          10824 non-null  int64
 3   Manufacturer  10824 non-null  str  
 4   Price         10824 non-null  int64
 5   Condition     10824 non-null  str  
 6   Cylinders     10824 non-null  int64
 7   Fuel          10824 non-null  str  
 8   Odometer      10824 non-null  int64
 9   Transmission  10824 non-null  str  
 10  Drive         10824 non-null  str  
 11  Type          10824 non-null  str  
 12  Paint         10824 non-null  str  
 13  State         10824 non-null  str  
dtypes: int64(5), str(9)
memory usage: 1.2 MB


In [4]:
df.isna().sum()

Model           0
Vehicle         0
Year            0
Manufacturer    0
Price           0
Condition       0
Cylinders       0
Fuel            0
Odometer        0
Transmission    0
Drive           0
Type            0
Paint           0
State           0
dtype: int64

In [5]:
df.describe()

,Vehicle,Year,Price,Cylinders,Odometer
count,10824.000000,10824.000000,10824.000000,10824.000000,1.082400e+04
mean,10793.513211,2008.518016,14561.422025,5.927476,1.233869e+05
std,6257.525632,7.838022,12949.364710,1.615951,8.674453e+04
min,1.000000,1970.000000,500.000000,3.000000,5.000000e+00
25%,5341.750000,2005.000000,5900.000000,4.000000,7.800000e+04
50%,10818.000000,2010.000000,10000.000000,6.000000,1.180000e+05
75%,16211.500000,2014.000000,18995.000000,8.000000,1.597302e+05
max,21648.000000,2022.000000,150000.000000,8.000000,3.333333e+06


In [6]:
# get string cols
str_cols = df.select_dtypes(include="str").columns
print(list(str_cols))

['Model', 'Manufacturer', 'Condition', 'Fuel', 'Transmission', 'Drive', 'Type', 'Paint', 'State']


In [7]:
# distinct values and counts for each one
for col in str_cols:
    print(f"--- {col} ({df[col].nunique()} distinct) ---")
    print(df[col].value_counts(dropna=False))
    print()

--- Model (3128 distinct) ---
Model
f-150                       169
silverado 1500              129
accord                      116
camry                       104
1500                         95
                           ... 
tuarus                        1
fusion titanium hybrid        1
ltd 2dr cpe                   1
srx4 awd                      1
hardtop cooper hatchback      1
Name: count, Length: 3128, dtype: int64

--- Manufacturer (38 distinct) ---
Manufacturer
ford               1999
chevrolet          1398
toyota             1117
honda               725
nissan              616
bmw                 391
ram                 388
mercedes-benz       366
gmc                 356
jeep                333
dodge               312
volkswagen          280
hyundai             258
subaru              242
lexus               240
audi                180
kia                 173
mazda               158
chrysler            158
cadillac            142
mitsubishi          125
acura              

In [ ]:
# feature engineering section
# Model - 3128 Dist - drop... with 10000 rows total and 3128 values it doesn't make sense to include this given that
# we have other features (manufacturer, type, etc.) that should give us good signal

# Manufacturer - 38 distinct - classify into tiers of luxury, also classify into domestic vs foreign

# Condition - 5 distinct - these can be 0, 1, 2, 3, 4 etc because they go from older to newer

# Fuel - 4 distinct - fine to make into dummy variable

# Transmission - 2 distinct - fine to make into manual binary indicator

# Drive - 3 distinct - make dummy variable

# Type - 9 distinct - make dummy variable

# Paint - 12 distinct - make dummy variable

# State - 51 distinct - make into cost of living (low, medium, high) variable and make into region variable (southwest, northeast, midwest, southeast)



In [ ]:
# feature engineering mapping and function generated via Claude per advice given in class:
# lookup tables (fixed rules, nothing learned from the data)
TIER_MAP = {
    **{m: 0 for m in ["ford", "chevrolet", "toyota", "honda", "nissan", "ram", "gmc", "jeep",
                      "dodge", "volkswagen", "hyundai", "subaru", "kia", "mazda", "chrysler",
                      "mitsubishi", "buick", "pontiac", "mercury", "saturn", "fiat", "datsun",
                      "harley-davidson"]},
    **{m: 1 for m in ["bmw", "mercedes-benz", "audi", "lexus", "acura", "infiniti", "volvo",
                      "cadillac", "lincoln", "mini", "jaguar", "rover", "alfa-romeo"]},
    **{m: 2 for m in ["porsche", "ferrari"]},
}
DOMESTIC = {"ford", "chevrolet", "ram", "gmc", "jeep", "dodge", "chrysler", "cadillac",
            "buick", "lincoln", "pontiac", "mercury", "saturn", "harley-davidson"}
CONDITION_MAP = {"fair": 0, "good": 1, "excellent": 2, "like new": 3, "new": 4}

HIGH_COL = {"ca", "hi", "dc", "ny", "ma", "nj", "ct", "wa", "md", "ri", "nh", "vt", "or", "ak"}
LOW_COL = {"al", "ar", "ms", "ok", "ks", "mo", "ia", "in", "oh", "ky", "tn", "wv", "ne",
           "nd", "sd", "la", "mi", "wi"}
REGION_MAP = {
    **{s: "northeast" for s in ["ct", "me", "ma", "nh", "ri", "vt", "nj", "ny", "pa"]},
    **{s: "southeast" for s in ["al", "ar", "fl", "ga", "ky", "la", "ms", "nc", "sc", "tn",
                                "va", "wv", "de", "md", "dc"]},
    **{s: "southwest" for s in ["az", "nm", "ok", "tx"]},
    **{s: "midwest" for s in ["il", "in", "ia", "ks", "mi", "mn", "mo", "ne", "nd", "oh", "sd", "wi"]},
    **{s: "west" for s in ["ak", "ca", "co", "hi", "id", "mt", "nv", "or", "ut", "wa", "wy"]},
}
DUMMY_COLS = ["Fuel", "Drive", "Type", "Paint", "Region"]


def engineer_features(data, columns=None):
    """Turn raw palanquin rows into a numeric feature table.

    columns: pass X_train.columns when transforming new data (test set or a single
             car) so the result has exactly the training columns, in the same order.
    """
    out = data.copy()

    # tidy the text columns we map on
    for c in ["Manufacturer", "Condition", "Transmission", "State"]:
        out[c] = out[c].str.strip().str.lower()

    # Manufacturer: luxury tier and domestic indicator
    out["Luxury_Tier"] = out["Manufacturer"].map(TIER_MAP)
    out["Domestic"] = out["Manufacturer"].isin(DOMESTIC).astype(int)

    # Condition: ordinal
    out["Condition_num"] = out["Condition"].map(CONDITION_MAP)

    # Transmission: manual indicator
    out["Manual"] = (out["Transmission"] == "manual").astype(int)

    # State: cost-of-living tier and region
    out["Cost_of_Living"] = out["State"].map(
        lambda s: 2 if s in HIGH_COL else (0 if s in LOW_COL else 1)
    )
    out["Region"] = out["State"].map(REGION_MAP)

    # fail loudly if any lookup missed
    check = out[["Luxury_Tier", "Condition_num", "Region"]].isna().sum()
    if check.any():
        raise ValueError(f"Unmapped values found:\n{check[check > 0]}")

    # dummies, then drop the original string columns
    out = pd.get_dummies(out, columns=DUMMY_COLS, dtype=int)
    out = out.drop(columns=["Model", "Manufacturer", "Condition", "Transmission", "State"],
                   errors="ignore")

    # match the training columns (adds missing dummies as 0, fixes order, drops extras)
    if columns is not None:
        out = out.reindex(columns=columns, fill_value=0)

    return out

In [10]:
df_fe = engineer_features(df)

df_fe.head()

,Vehicle,Year,Price,Cylinders,Odometer,Luxury_Tier,Domestic,Condition_num,Manual,Cost_of_Living,...,Paint_purple,Paint_red,Paint_silver,Paint_white,Paint_yellow,Region_midwest,Region_northeast,Region_southeast,Region_southwest,Region_west
0,8680,2016,33995,6,48431,0,1,1,0,2,...,0,0,0,0,0,0,0,0,0,1
1,10426,2018,29000,4,17622,1,0,3,0,1,...,0,0,0,0,0,0,0,0,0,1
2,7243,1994,3950,8,222000,0,1,3,0,2,...,0,0,0,1,0,0,0,0,0,1
3,6225,2012,10900,8,112000,0,0,3,0,2,...,0,0,1,0,0,0,0,0,0,1
4,4501,2001,8300,6,264000,0,0,2,0,2,...,0,1,0,0,0,0,0,0,0,1


In [11]:
# split data
train_df, test_df = train_test_split(df_fe, test_size=0.5, random_state=42)

# drop the vehicle ID column if it exists
train_df = train_df.drop(columns=["Vehicle ID"], errors="ignore")
test_df = test_df.drop(columns=["Vehicle ID"], errors="ignore")

In [12]:
# prep for modeling

X_train = train_df.drop(columns="Price")
y_train = train_df["Price"]
X_test = test_df.drop(columns="Price")
y_test = test_df["Price"]

## Part A

In [15]:
cv = KFold(n_splits=10, shuffle=True, random_state=42)

full_tree = DecisionTreeRegressor(random_state=42)
full_rmse = -cross_val_score(full_tree, X_train, y_train, cv=cv,
                             scoring="neg_root_mean_squared_error").mean()
print("Full tree CV RMSE:", full_rmse)

Full tree CV RMSE: 8093.7177697670895


In [16]:
full_tree.fit(X_train, y_train)
print("Full tree leaves:", full_tree.get_n_leaves(), "depth:", full_tree.get_depth())

Full tree leaves: 4886 depth: 32


In [19]:
path = DecisionTreeRegressor(random_state=42).cost_complexity_pruning_path(X_train, y_train)

# examining alphas code generated with help from Claude
alphas = path.ccp_alphas[:-1]       # drop the last: it prunes down to the root
print(f"Number of alphas: {len(alphas)}, min: {alphas.min()}, max: {alphas.max()}")

Number of alphas: 4381, min: 0.0, max: 17752432.73166374


In [20]:
grid_alphas = np.logspace(np.log10(alphas[alphas > 0].min()), np.log10(alphas.max()), 40)
print(f"Grid alphas: {grid_alphas}")

Grid alphas: [9.23872875e-05 1.79860777e-04 3.50155306e-04 6.81686918e-04
 1.32711698e-03 2.58364866e-03 5.02988091e-03 9.79223777e-03
 1.90636562e-02 3.71133746e-02 7.22528019e-02 1.40662697e-01
 2.73843974e-01 5.33123021e-01 1.03789085e+00 2.02057945e+00
 3.93369043e+00 7.65815984e+00 1.49090055e+01 2.90250464e+01
 5.65063389e+01 1.10007278e+02 2.14163603e+02 4.16936497e+02
 8.11697411e+02 1.58022311e+03 3.07639896e+03 5.98917363e+03
 1.16598013e+04 2.26994534e+04 4.41915919e+04 8.60327677e+04
 1.67489714e+05 3.26071161e+05 6.34799590e+05 1.23583612e+06
 2.40594188e+06 4.68391905e+06 9.11871470e+06 1.77524327e+07]


In [21]:
tree_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid={"ccp_alpha": grid_alphas},
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
tree_search.fit(X_train, y_train)

print("Best ccp_alpha:", tree_search.best_params_["ccp_alpha"])
print("Pruned CV RMSE:", -tree_search.best_score_)
print("Full CV RMSE:  ", full_rmse)
print("Pruned leaves: ", tree_search.best_estimator_.get_n_leaves())

Best ccp_alpha: 86032.7676560414
Pruned CV RMSE: 7478.562978527341
Full CV RMSE:   8093.7177697670895
Pruned leaves:  140


## Part B

In [23]:
new_car = pd.DataFrame([{
    "Year": 2016,
    "Manufacturer": "dodge",
    "Model": "charger",
    "Condition": "like new",
    "Cylinders": 6,
    "Fuel": "gas",
    "Odometer": 62000,
    "Transmission": "automatic",
    "Drive": "rwd",
    "Type": "sedan",
    "Paint": "silver",
    "State": "al",
}])

new_car_fe = engineer_features(new_car, columns=X_train.columns)

pred = tree_search.best_estimator_.predict(new_car_fe)
print(pred)

[21390.5]


## Part C

In [27]:
# Random Forest on training data
rf_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid={
        # "n_estimators": [100, 300, 500], - tried this but RMSE was 5496 with 500 estimators and took way longer
        "n_estimators": [100, 300],
        "max_features": [0.2, 0.33, 0.5, 0.75, 1.0],
        "min_samples_leaf": [1, 2, 5],
    },
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)

print("Best Parameters", rf_search.best_params_)
print("Best CV RMSE", -rf_search.best_score_)

Best Parameters {'max_features': 0.33, 'min_samples_leaf': 1, 'n_estimators': 300}
Best CV RMSE 5500.139827711852


## Part D

In [30]:
# Gradient Boosting on training data

# I did try swapping in more estimators and smaller learning rates, but it didn't improve the RMSE significantly,
# The RMSE I got was 5616.731112225292 so I stuck with the current one that isn't commented out
    # param_grid={
    #     "n_estimators": [800, 1200, 1600],
    #     "learning_rate": [0.01, 0.02, 0.05],
    #     "max_depth": [4, 5, 6],
    # },
gb_search = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid={
        "n_estimators": [200, 400, 800],
        "learning_rate": [0.02, 0.05, 0.1],
        "max_depth": [3, 4, 5, 6],
    },
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
gb_search.fit(X_train, y_train)

print("Best Parameters", gb_search.best_params_)
print("Best CV RMSE", -gb_search.best_score_)

Best Parameters {'learning_rate': 0.02, 'max_depth': 5, 'n_estimators': 800}
Best CV RMSE 5625.325449982929


## Part E

In [39]:
# fit the final model and evaluate on the test set
final_model = rf_search.best_estimator_
test_preds = final_model.predict(X_test)

test_rmse = root_mean_squared_error(y_test, test_preds)
print("RF CV RMSE:  ", -rf_search.best_score_)
print("RF Test RMSE:", test_rmse)
print("RF Mean price:", y_test.mean())

RF CV RMSE:   5500.139827711852
RF Test RMSE: 6450.283194968927
RF Mean price: 14723.509053954176


In [40]:
# That seems like a really large jump between train and test performance. Can I dive into the CV scores to see if these were also really high?
fold_rmse = -cross_val_score(rf_search.best_estimator_, X_train, y_train,
                             cv=cv, scoring="neg_root_mean_squared_error")
print(fold_rmse.round(0), fold_rmse.mean(), fold_rmse.std())

[4769. 6796. 5148. 7828. 4782. 5601. 4475. 5196. 4841. 5567.] 5500.139827711852 993.1512248904153


In [ ]:
# kind of cheating to look at test, but this describes some variance here that max is 150k in test and 129k in train
print(y_train.describe(), y_test.describe())

count      5412.000000
mean      14399.334996
std       12529.500350
min         500.000000
25%        5900.000000
50%        9999.000000
75%       18999.000000
max      129900.000000
Name: Price, dtype: float64 count      5412.000000
mean      14723.509054
std       13355.229294
min         500.000000
25%        5900.000000
50%       10499.500000
75%       18995.000000
max      150000.000000
Name: Price, dtype: float64
